# 🔬 τ-Knowledge 소스 준비 (고급 사용자/운영자용)

이 노트북은 **저작 경로 (authoring path)**의 첫 번째 단계입니다.  
학습자 노트북(03, 04)에서 사용하는 **준비된 데이터 번들**을 만들기 위한 소스를 준비합니다.

## 대상
- 운영자 / 고급 사용자 / 데이터 엔지니어
- SDG(합성 데이터 생성)를 직접 수행하려는 사용자

## 처리 단계

1. τ-bench 버전 고정 및 KB(지식 기반) 검사
2. 정책 사실(facts) 추출
3. 도구 스키마 검사
4. 공식 태스크/분할 경계 검토
5. 소스 스냅샷 저장

### 핵심 원칙
- **평가 비공개 정보**(기대 행동, 보상 기준, 숨겨진 사용자 목표)는 학습 데이터에 포함하지 않습니다
- 공식 평가 태스크는 평가용으로 보존하고, 독립적인 학습 시나리오를 생성합니다
- 모든 소스에 대해 버전, SHA, 라이선스를 기록합니다

In [ ]:
"""Pin τ-bench version and inspect KB."""

import os
import json
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_yaml_config, PROJECT_ROOT,
)

load_env()

# Load preparation config
prep_config = load_yaml_config("configs/data-preparation.yaml")
tau_config = prep_config["tau_bench"]

tau_version = tau_config.get("version", os.environ.get("TAU_BENCH_VERSION", ""))
tau_sha = tau_config.get("commit_sha", os.environ.get("TAU_BENCH_COMMIT_SHA", ""))
tau_domain = tau_config.get("domain", "banking_knowledge")

print("=" * 70)
print("📌 τ-bench 버전 고정")
print("=" * 70)
print(f"  버전: {tau_version}")
print(f"  커밋 SHA: {tau_sha or '(미설정 — 검증 후 고정 필요)'}")
print(f"  도메인: {tau_domain}")
print()

# Check τ-bench installation
tau_install_path = tau_config.get("install_path", os.environ.get("TAU_BENCH_INSTALL_PATH", ""))

if tau_install_path and Path(tau_install_path).exists():
    print(f"✅ τ-bench 설치 경로: {tau_install_path}")

    # Inspect KB
    kb_paths = list(Path(tau_install_path).rglob("*knowledge*"))
    print(f"\n   KB 관련 파일:")
    for p in kb_paths[:10]:
        print(f"     {p.relative_to(tau_install_path)}")
else:
    print("⚠️  τ-bench 설치 경로가 설정되지 않았거나 존재하지 않습니다.")
    print("  TAU_BENCH_INSTALL_PATH를 .env에 설정하세요.")
    print("  또는: pip install tau2-bench")
    print()
    print("  참조: https://github.com/sierra-research/tau2-bench")

# Verify v1.0.1 grading correction
print("\n📋 버전 참고사항:")
print("  v1.0.1에는 banking 채점 수정이 포함되어 있습니다.")
print("  수정 전후의 점수는 직접 비교할 수 없습니다.")

In [ ]:
"""Extract policy facts from KB."""

source_config = prep_config["sources"]
kb_output = PROJECT_ROOT / source_config["kb_snapshot"]["output_path"]
policy_output = PROJECT_ROOT / source_config["policy_extraction"]["output_path"]

print("=" * 70)
print("📋 정책 사실 추출")
print("=" * 70)

kb_output.parent.mkdir(parents=True, exist_ok=True)
policy_output.parent.mkdir(parents=True, exist_ok=True)

if tau_install_path and Path(tau_install_path).exists():
    # Load KB documents
    print("KB 문서 로딩 중...")

    try:
        # Try loading from τ-bench API
        import sys
        sys.path.insert(0, tau_install_path)

        # Inspect KB structure
        print("\n  KB 구조 검사:")
        print("    - 문서 경계 보존")
        print("    - 섹션, 링크, 정책 버전 유지")
        print("    - 조건 및 예외 사항 포함")

        # Extract policy facts
        print("\n  정책 사실 추출:")
        print(f"    검토 샘플 크기: {source_config['policy_extraction']['review_sample_size']}")
        print(f"    소스 오프셋 보존: {source_config['policy_extraction']['retain_source_offsets']}")
        print(f"    문서 ID 보존: {source_config['policy_extraction']['retain_document_ids']}")

        # Note: actual extraction requires τ-bench APIs
        print("\n  ℹ️  실제 추출은 τ-bench API를 통해 수행됩니다.")
        print("     수동 실행:")
        print(f"     python scripts/prepare_tau_sources.py --config configs/data-preparation.yaml")

    except Exception as exc:
        print(f"\n  ⚠️  KB 로드 실패: {exc}")
        print("     τ-bench 설치를 확인하세요.")
else:
    print("⚠️  τ-bench 미설치 — 정책 추출을 수행할 수 없습니다.")
    print("   수동 실행: python scripts/prepare_tau_sources.py")

In [ ]:
"""Inspect tool schemas."""

print("=" * 70)
print("🔧 도구 스키마 검사")
print("=" * 70)

if tau_install_path and Path(tau_install_path).exists():
    # Look for tool definitions
    tool_files = list(Path(tau_install_path).rglob("*tool*"))
    tool_files += list(Path(tau_install_path).rglob("*action*"))

    print(f"  도구 관련 파일: {len(tool_files)}개")
    for tf in tool_files[:15]:
        print(f"    {tf.relative_to(tau_install_path)}")

    print("\n  검사 항목:")
    print("    ✓ 도구 이름 및 설명")
    print("    ✓ 매개변수 스키마 (JSON Schema)")
    print("    ✓ 필수/선택 매개변수")
    print("    ✓ 반환 값 형식")
    print("    ✓ 상태 변경 도구 식별")
    print("    ✓ 도구 발견 동작 (공식 행동 보존)")
else:
    print("⚠️  τ-bench 미설치 — 도구 스키마 검사를 수행할 수 없습니다.")

print("\n  ⚠️  공식 도구 발견 동작을 보존해야 합니다.")
print("     편의상 모든 도구 스키마를 미리 노출하는 경우,")
print("     수정된 벤치마크 조건으로 표기해야 합니다.")

In [ ]:
"""Review official task/split boundaries."""

print("=" * 70)
print("📊 공식 태스크/분할 경계 검토")
print("=" * 70)

split_config = prep_config["splits"]

print(f"  분할 방법: {split_config['method']}")
print(f"  학습 비율: {split_config['train_ratio']}")
print(f"  검증 비율: {split_config['validation_ratio']}")
print(f"  시드: {split_config['seed']}")
print(f"  오염 검사: {split_config['contamination_check']}")
print(f"  패밀리 격리: {split_config['family_isolation']}")

print("\n📋 분할 원칙:")
print("  1. 공식 평가 태스크는 모두 평가용으로 보존")
print("  2. 공식 학습 분할이 있으면 허용 조건 확인 후 사용")
print("  3. 없으면 KB와 독립 시나리오에서 학습/검증 데이터 생성")
print("  4. 패러프레이즈, 이름/번호 치환, 형제 예제는 같은 분할에 유지")
print("  5. 평가 태스크 문구, 기대 행동, 골든 문서 목록은 절대 SDG에 사용 불가")

print("\n⚠️  핵심 구분:")
print("  • kb_adaptation: KB를 학습하고 새 상황에 적용 → 본 실험")
print("  • 태스크 누출(leakage): 평가 태스크/시나리오를 학습에 사용 → 금지")
print("  • KB 공유는 kb_adaptation에서 의도적이며, 태스크 누출과 구별됨")

In [ ]:
"""Save source snapshot."""

output_base = PROJECT_ROOT / prep_config["output"]["base_path"]
output_base.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("💾 소스 스냅샷 저장")
print("=" * 70)

# Save snapshot metadata
snapshot_meta = {
    "tau_version": tau_version,
    "tau_commit_sha": tau_sha,
    "domain": tau_domain,
    "split_config": split_config,
    "quality_gates": prep_config.get("quality_gates", {}),
    "output_paths": {
        "kb_snapshot": str(kb_output),
        "policy_extraction": str(policy_output),
    },
}

meta_path = output_base / "snapshot_metadata.json"
with open(meta_path, "w") as f:
    json.dump(snapshot_meta, f, indent=2, ensure_ascii=False)

print(f"✅ 스냅샷 메타데이터 저장: {meta_path}")
print(f"   출력 디렉토리: {output_base}")

# Show quality gates
quality_gates = prep_config.get("quality_gates", {})
print("\n📊 품질 게이트:")
print(f"  최소 수용률: {quality_gates.get('min_acceptance_rate', 'N/A')}")
print(f"  필수 유형: {quality_gates.get('required_types', [])}")
print(f"  최소 도구 예제: {quality_gates.get('min_tool_examples', 'N/A')}")
print(f"  최소 궤적 예제: {quality_gates.get('min_trajectory_examples', 'N/A')}")
print(f"  인적 검토 샘플: {quality_gates.get('human_review_sample_size', 'N/A')}")

print("\n다음 단계:")
print("  📓 02_generate_synthetic.ipynb — 합성 데이터 생성")
print("  또는 CLI: python scripts/prepare_tau_sources.py --config configs/data-preparation.yaml")